# Hugging Face Inference API: Quick Start

**Day 1 Afternoon - Warm-up Exercise**

---

## Overview

This notebook demonstrates how to use the **Hugging Face Inference API** to run genomic foundation models **without downloading them locally**. This is useful for:

- Quick prototyping and testing
- Running models when you don't have a GPU
- Accessing models without managing dependencies

We will go through the code **step by step**, explaining each part so you understand how the different components work together.

## What We'll Do

1. Set up authentication with HF API token
2. Configure the Nucleotide Transformer endpoint
3. Send a masked DNA sequence for prediction
4. Interpret the model's predictions

## Prerequisites

- A Hugging Face account ([huggingface.co](https://huggingface.co))
- An API token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
- Token stored in environment variable `HF_TOKEN`

---

In [ ]:
# Required libraries
# - requests: For making HTTP calls to the HF Inference API
# - os: For accessing environment variables (API token)
import requests 
import os

print("Libraries imported successfully!")

In [ ]:
# Load API token from environment variable
# To set this: export HF_TOKEN="your_token_here" (in terminal)
# Or in notebook: os.environ['HF_TOKEN'] = 'your_token_here'
#API_TOKEN = os.getenv('HF_TOKEN')
API_TOKEN = "YOUR_HF_TOKEN_HERE"

# Verify token is loaded
if API_TOKEN:
    print(f"API token loaded successfully (starts with: {API_TOKEN[:8]}...)")
else:
    print("WARNING: HF_TOKEN not found. Set it with: export HF_TOKEN='your_token'")

## Step 1: Authentication

The Inference API requires authentication via an API token. You can get your token from:
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

**Security tip:** Never hardcode tokens in notebooks. Use environment variables instead.

In [ ]:
# Define the model to use
# This is the Nucleotide Transformer trained on the human reference genome
# Other options: nucleotide-transformer-500m-1000g, nucleotide-transformer-2.5b-multi-species
model_id = "InstaDeepAI/nucleotide-transformer-500m-human-ref"

# Construct the API endpoint URL
API_URL = f"https://router.huggingface.co/hf-inference/models/{model_id}"

# Set up authentication headers
headers = {"Authorization": f"Bearer {API_TOKEN}"}

print(f"Model: {model_id}")
print(f"Endpoint: {API_URL}")

## Step 2: Configure the Model Endpoint

We'll use the **Nucleotide Transformer** (500M parameters), a genomic foundation model trained on the human reference genome.

The Inference API URL format is:
```
https://router.huggingface.co/hf-inference/models/{model_id}
```

In [ ]:
# Create a DNA sequence with a masked position
# The <mask> token tells the model: "predict what goes here"
masked_sequence = "ACGTGCAC<mask>GGACCAGCA"

# The payload is a simple JSON object with the input sequence
payload = {"inputs": masked_sequence}

print(f"Input sequence: {masked_sequence}")
print(f"The model will predict what 6-mer should replace <mask>")

## Step 3: Prepare the Input

The Nucleotide Transformer is trained with **masked language modeling (MLM)**.
We provide a DNA sequence with a `<mask>` token, and the model predicts what nucleotides should fill that position.

This is similar to how BERT predicts masked words in sentences.

In [ ]:
# Send POST request to the Inference API
# - API_URL: The model endpoint
# - headers: Contains our authentication token
# - json: The input data (our masked sequence)
response = requests.post(API_URL, headers=headers, json=payload)

# Check if the request was successful
# 200 = Success, 503 = Model loading, 401 = Auth error
print(f"Status code: {response.status_code}")

if response.status_code == 200:
    print("Success! Model returned predictions.")
elif response.status_code == 503:
    print("Model is loading. Wait a moment and try again.")
else:
    print(f"Error: {response.text}")

## Step 4: Call the API

Now we send our masked sequence to the Inference API. The model runs on Hugging Face's servers and returns predictions.

In [ ]:
# Parse and display the top predictions
if response.status_code == 200:
    # Parse JSON response
    results = response.json()
    
    print("Top 5 predictions for <mask>:")
    print("-" * 40)
    
    # Display each prediction with its confidence score
    for i, pred in enumerate(results[:5]):
        token = pred['token_str']      # The predicted 6-mer
        score = pred['score'] * 100    # Convert to percentage
        print(f"  {i+1}. '{token}' - {score:.1f}% confidence")
    
    print("-" * 40)
    print(f"\nThe model predicts '{results[0]['token_str']}' as the most likely 6-mer at the masked position.")
else:
    print("No results to display. Check the error above.")

## Step 5: Interpret the Results

The API returns a list of predictions, each with:
- `token_str`: The predicted token (6-mer in this case)
- `score`: Probability/confidence of this prediction

---

## Summary

In this notebook, you learned how to:

| Step | What We Did |
|------|-------------|
| **Authentication** | Set up HF API token from environment variable |
| **Endpoint Setup** | Configure the Nucleotide Transformer model URL |
| **Input Preparation** | Create a masked DNA sequence |
| **API Call** | Send request and handle response |
| **Interpretation** | Parse and display model predictions |

## Key Takeaways

- The **Inference API** lets you run models without downloading them
- **Masked Language Modeling** predicts missing tokens in sequences
- The Nucleotide Transformer predicts **6-mers** (6 nucleotide chunks)
- Response includes **multiple predictions** with confidence scores

## Next Steps

- Try different masked sequences
- Explore other genomic models on [HuggingFace Hub](https://huggingface.co/models?other=dna)
- In the next notebooks, we'll download models locally for more control

---

**Congratulations!** You've successfully used the HuggingFace Inference API with a genomic foundation model!